# Self-Supervised Objectives: SimCLR's NT-Xent and DINO

**Difficulty**: 🟡 Medium

### Problem Statement

Most strong image encoders today are pretrained without labels. Two families of joint-embedding methods dominate: **contrastive learning** (SimCLR, MoCo) and **self-distillation** (BYOL, DINO, and DINOv2, whose frozen features are widely used for depth, segmentation and retrieval). Both make several random augmentations ("views") of every image and train the encoder so that the views of one image agree. They differ in how they avoid the trivial solution: an encoder that maps every image to the same output.

**SimCLR** (Chen et al., ICML 2020) uses **negatives**. A batch of $N$ images gives $2N$ views $z_1, \dots, z_{2N}$ (here, the rows of `cat([z1, z2])`). Each view must pick out the other view of its own image among all $2N - 1$ other views. This is the **NT-Xent** loss (normalised temperature-scaled cross-entropy), eq. (1) of the paper:

$$\ell(i, j) = -\log \frac{\exp(\mathrm{sim}(z_i, z_j)/\tau)}{\sum_{k=1}^{2N} \mathbb{1}_{[k \neq i]}\exp(\mathrm{sim}(z_i, z_k)/\tau)}, \qquad \mathrm{sim}(u, v) = \frac{u^\top v}{\lVert u\rVert\,\lVert v\rVert}, \qquad \mathcal L = \frac{1}{2N}\sum_{i=1}^{2N} \ell\big(i, \mathrm{pos}(i)\big)$$

Here $\mathrm{pos}(i)$ is the other view of the same image. Every one of the $2N$ views is an anchor, and each anchor has 1 positive and $2(N-1)$ negatives. Compare CLIP's InfoNCE (`v3/modern-architectures/contrastive-loss-clip`). CLIP has two modalities, so its logits are the $N \times N$ image–text block: the negatives of an image are only the $N-1$ other captions, and an image is never compared with itself. SimCLR's $2N$ views all live in one space. Its logit matrix is $2N \times 2N$, the negatives include views from the same half of the batch, and the diagonal (a view against itself, cosine always 1) must be **removed**.

**DINO** (Caron et al., ICCV 2021) has **no negatives**. A student network $g_s$ is trained to match a **teacher** $g_t$ that receives no gradients: its weights are an exponential moving average (EMA) of the student's. Both networks end in a $K$-way softmax head. With **multi-crop**, the teacher sees only the 2 **global** crops, while the student sees those plus several small **local** crops. Every student crop is trained to predict the teacher's distribution for each global crop *other than itself*. Two operations on the teacher's output prevent collapse. **Centering** subtracts a running mean $c$ of the teacher logits, and **sharpening** uses a low teacher temperature $\tau_t$ (0.04, against $\tau_s = 0.1$ for the student). With $V$ crops (crops 0 and 1 global), batch size $B$ and $H(p, q) = -\sum_k p_k \log q_k$:

$$P_t^{(i)} = \mathrm{softmax}\Big(\frac{g_t(x_i) - c}{\tau_t}\Big),\qquad P_s^{(v)} = \mathrm{softmax}\Big(\frac{g_s(x_v)}{\tau_s}\Big),\qquad \mathcal L = \frac{1}{2(V-1)}\sum_{i=0}^{1}\ \sum_{\substack{v=0\\ v\neq i}}^{V-1}\ \frac1B\sum_{b=1}^{B} H\big(P_t^{(i)}[b],\, P_s^{(v)}[b]\big)$$

After the loss, and without gradient, the center moves towards the batch mean of the teacher logits, $c \leftarrow m\,c + (1-m)\,\overline{g_t}$, and the teacher's weights move towards the student's, $\theta_t \leftarrow \lambda\,\theta_t + (1-\lambda)\,\theta_s$.

Implement both losses and the in-place EMA update of the teacher. (`v4/dl-fundamentals/training-utilities` builds a general-purpose EMA class that keeps a shadow copy of the weights. Here the EMA *is* the teacher network, updated in place as in DINO's training loop.)

---

### Requirements

1. **`nt_xent_loss(z1, z2, temperature=0.1) -> Tensor`**
   - `z1`, `z2`: `(N, D)` projection-head outputs, **not** normalised. Row `i` of `z1` and row `i` of `z2` are the two views of image `i`. $N \ge 1$.
   - L2-normalise, form the $(2N, 2N)$ matrix of cosine similarities between all views divided by `temperature`, exclude each anchor's similarity with itself, and use the view $N$ rows away as the positive. `temperature` is $\tau$ and **divides** the cosines, as in the SimCLR paper (the CLIP problem passed a multiplier instead).
   - Return the **mean over all $2N$ anchors** as a 0-dim tensor. This is the paper's average; the official TF code returns the *sum* of the two halves' means, which is twice this value.
   - Differentiable, does not modify its inputs, no Python loop over the batch, and finite for `temperature` as small as 0.005 in float32 (logits up to 200).
2. **`DINOLoss(out_dim, ncrops, student_temp=0.1, teacher_temp=0.04, center_momentum=0.9)`**, an `nn.Module` with:
   - a **buffer** (not a parameter) named `center` of shape `(1, out_dim)`, initialised to zeros. Use `register_buffer`, so it is saved in `state_dict` and follows `.to(device)`.
   - `forward(student_output, teacher_output) -> Tensor` (0-dim), where
     - `student_output` is `(ncrops * B, K)`: all crops stacked crop by crop, so rows `v*B : (v+1)*B` are crop `v`, and crops 0 and 1 are the two global crops. `teacher_output` is `(2 * B, K)`: the two global crops, in the same order. This is how DINO's `MultiCropWrapper` concatenates its outputs.
     - Teacher probabilities are `softmax((teacher_output - center) / teacher_temp)`, and **no gradient** may flow into `teacher_output` (it may arrive with `requires_grad=True`). Student log-probabilities are `log_softmax(student_output / student_temp)`.
     - For every pair (teacher crop `iq` ∈ {0, 1}, student crop `v`) with `v != iq`, take the batch mean of $H(P_t, P_s)$. Sum over the pairs and divide by the number of pairs, `2 * (ncrops - 1)`. This is the official averaging: `total_loss += loss.mean()` for each pair, then `total_loss /= n_loss_terms`.
     - **After** computing the loss, update the center under `torch.no_grad()`: `center = m * center + (1 - m) * teacher_output.mean(0, keepdim=True)`. The mean is over all `2B` rows of the raw (uncentred) teacher logits, and the loss of this call uses the center from before the update.
     - A loop over the (at most $2V$) crop pairs is fine. A loop over the samples is not.
   - Left out (not graded): the official teacher-temperature warm-up schedule, and the `all_reduce` of the batch center across GPUs.
3. **`ema_update_(teacher, student, momentum) -> None`**: for every pair of parameters of two modules with the same architecture, $\theta_t \leftarrow m\,\theta_t + (1 - m)\,\theta_s$, **in place**: the same storage is overwritten, so `p.data = ...` (which allocates a new tensor) does not count.
   - No autograd. Afterwards the teacher's parameters are still the same leaf `nn.Parameter` objects, with no `grad_fn` and unchanged `requires_grad`, whether or not they required grad before. The student (values and `.grad`) is untouched.
   - **Parameters only; leave buffers alone.** This is what DINO's training loop does. Buffers such as BatchNorm running statistics are not trained by gradient descent; they describe the activations of the network that produced them. DINO never switches the teacher to eval mode, so a BatchNorm teacher normalises with its own batch statistics and keeps its own running stats. (The EMA in `training-utilities` copies buffers instead, because its averaged weights replace the model's at evaluation time.)

---

### Constraints

- ✅ Built-in ops such as `F.normalize`, `F.cross_entropy`, `F.softmax`, `F.log_softmax` and `torch.logsumexp`.
- ✅ The brute-force loops in the setup cell are reference oracles for the tests; don't call them in your implementation.
- ❌ No Python loop over the samples of the batch, in either loss.
- ❌ No `torch.log(torch.softmax(...))`, and no `exp()` followed by a division. At the temperatures used in practice these overflow or underflow.
- ❌ No `.item()` or `float(...)` inside the losses; they cut the autograd graph.

---

<details>
  <summary>💡 Hint</summary>

  - Stack the two views into one $(2N, D)$ batch. Each row of the $2N \times 2N$ logit matrix is then a classification over the other $2N - 1$ views. What is the target class of row $i$? Of row $i + N$?
  - Which value makes an entry disappear from a softmax, 0 or $-\infty$?
  - `F.cross_entropy` with the default `reduction="mean"` already averages over the rows.
  - DINO: `tensor.chunk(n)` splits the stacked crops apart again. How many (teacher crop, student crop) pairs are left once you skip the same view?
  - Where does the center enter: the teacher's logits, the teacher's probabilities, or the student's logits? Whose outputs does it average?
  - The EMA update is not part of the computation graph. Which context manager keeps autograd out of it, and which in-place ops keep the teacher's `Parameter` objects alive?

</details>

---

In [ ]:
import copy
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# Two "augmented views" of N images: the same latent vector plus independent noise. The row
# norms differ a lot on purpose: the loss must only look at directions.
torch.manual_seed(0)
N, D = 8, 16
images = torch.randn(N, D)
z1 = (images + 0.3 * torch.randn(N, D)) * (torch.rand(N, 1) * 4 + 0.5)
z2 = (images + 0.3 * torch.randn(N, D)) * (torch.rand(N, 1) * 4 + 0.5)
print("z1", tuple(z1.shape), " z2", tuple(z2.shape))


def nt_xent_loop(z1, z2, tau):
    # Reference oracle: SimCLR eq. (1), one anchor at a time, in plain Python floats.
    z = torch.cat([z1, z2]).double()
    z = (z / z.norm(dim=1, keepdim=True)).tolist()
    n2 = len(z)
    total = 0.0
    for i in range(n2):
        j = (i + n2 // 2) % n2                                   # the other view of image i
        logits = {k: sum(a * b for a, b in zip(z[i], z[k])) / tau for k in range(n2) if k != i}
        top = max(logits.values())
        log_denominator = top + math.log(sum(math.exp(v - top) for v in logits.values()))
        total += log_denominator - logits[j]
    return total / n2


def dino_loop(student_output, teacher_output, center, ncrops, tau_s, tau_t):
    # Reference oracle: the DINO loss one (teacher crop, student crop, sample) term at a time.
    s, t, c = student_output.double(), teacher_output.double(), center.double().reshape(-1)
    B = t.shape[0] // 2
    total, terms = 0.0, 0
    for iq in range(2):                                          # the teacher sees global crops only
        for v in range(ncrops):
            if v == iq:
                continue                                         # never compare a crop with itself
            for b in range(B):
                p_t = torch.softmax((t[iq * B + b] - c) / tau_t, dim=0)
                log_p_s = torch.log_softmax(s[v * B + b] / tau_s, dim=0)
                total += -(p_t * log_p_s).sum().item()
                terms += 1
    return total / terms

In [ ]:
def nt_xent_loss(z1: torch.Tensor, z2: torch.Tensor, temperature: float = 0.1) -> torch.Tensor:
    """
    SimCLR's NT-Xent loss (Chen et al., 2020, eq. 1), averaged over all 2N anchors.

    Args:
        z1: (N, D) embeddings of the first view, not normalised.
        z2: (N, D) embeddings of the second view; row i of z1 and z2 come from image i.
        temperature: tau; the cosine similarities are DIVIDED by it.

    Returns:
        0-dim tensor: mean over the 2N views of -log softmax(positive) over the 2N - 1 other views.
    """
    # TODO: normalise, (2N, 2N) cos / tau, mask the diagonal with -inf, targets i <-> i + N,
    #       one cross-entropy
    raise NotImplementedError

In [ ]:
loss = nt_xent_loss(z1, z2, 0.5)
want = nt_xent_loop(z1, z2, 0.5)
assert torch.is_tensor(loss) and loss.dim() == 0, "return a 0-dim tensor"
assert abs(loss.item() - want) < 1e-5, f"got {loss.item():.6f}, expected {want:.6f}"

# Every view is an anchor, so swapping the two view batches cannot change the loss.
assert torch.allclose(nt_xent_loss(z2, z1, 0.5), loss, atol=1e-6), "only one direction of anchors?"
# Cosine similarity: the length of an embedding does not matter.
assert torch.allclose(nt_xent_loss(7.0 * z1, 0.01 * z2, 0.5), loss, atol=1e-5), "L2-normalise first"
# N = 1: once the anchor itself is excluded, its positive is the only candidate, so the loss is 0.
assert abs(nt_xent_loss(z1[:1], z2[:1], 0.5).item()) < 1e-6, "the diagonal is still in the softmax"
# A huge temperature makes every logit ~0: each anchor is uniform over its 2N - 1 candidates.
assert abs(nt_xent_loss(z1, z2, 1e6).item() - math.log(2 * N - 1)) < 1e-4
# A tiny temperature in float32: logits up to 200 must not overflow.
assert torch.isfinite(nt_xent_loss(z1, z2, 0.005)), "use cross_entropy / logsumexp, not exp()"

g1, g2 = z1.clone().requires_grad_(), z2.clone().requires_grad_()
nt_xent_loss(g1, g2, 0.1).backward()
assert g1.grad is not None and torch.isfinite(g1.grad).all() and torch.isfinite(g2.grad).all()
print("nt_xent_loss tests passed")
for tau in (0.05, 0.1, 0.5, 1.0):
    print(f"  tau = {tau:<4}  loss = {nt_xent_loss(z1, z2, tau).item():.4f}")
print(f"  chance level log(2N - 1) = {math.log(2 * N - 1):.4f}")

In [ ]:
class DINOLoss(nn.Module):
    """DINO's self-distillation loss with multi-crop, teacher centering and sharpening."""

    def __init__(self, out_dim: int, ncrops: int, student_temp: float = 0.1,
                 teacher_temp: float = 0.04, center_momentum: float = 0.9):
        super().__init__()
        # TODO: store the hyper-parameters; register a (1, out_dim) zero buffer named "center"
        raise NotImplementedError

    def forward(self, student_output: torch.Tensor, teacher_output: torch.Tensor) -> torch.Tensor:
        """
        Args:
            student_output: (ncrops * B, K) student logits, crop v in rows v*B:(v+1)*B,
                            crops 0 and 1 are the global crops.
            teacher_output: (2 * B, K) teacher logits for the two global crops (same order).

        Returns:
            0-dim tensor: mean over the 2 * (ncrops - 1) pairs (teacher crop iq, student crop v != iq)
            of the batch-mean cross-entropy H(P_t, P_s). Updates self.center afterwards.
        """
        # TODO: teacher probs from detached, centred, sharpened logits; student log_softmax;
        #       loop over crop pairs skipping v == iq; average; then update the center
        raise NotImplementedError

In [ ]:
torch.manual_seed(1)
B, K, n_local = 4, 10, 3
ncrops = 2 + n_local
student_out = torch.randn(ncrops * B, K)                               # all crops, crop by crop
teacher_out = 0.3 * torch.randn(2 * B, K) + torch.linspace(-1, 1, K)   # the 2 global crops
c0 = 0.1 * torch.randn(1, K)                                           # as if mid-training

dino = DINOLoss(K, ncrops)          # student_temp=0.1, teacher_temp=0.04, center_momentum=0.9
assert "center" in dict(dino.named_buffers()), "center must be a registered buffer"
assert dino.center.shape == (1, K) and (dino.center == 0).all(), "center starts at zeros, shape (1, K)"
with torch.no_grad():
    dino.center.copy_(c0)

s = student_out.clone().requires_grad_()
t = teacher_out.clone().requires_grad_()     # the grader may also hand you a teacher with grad
loss = dino(s, t)
want = dino_loop(student_out, teacher_out, c0, ncrops, 0.1, 0.04)
assert loss.dim() == 0 and abs(loss.item() - want) < 1e-5 * max(1.0, want), (loss.item(), want)

loss.backward()
assert t.grad is None or (t.grad == 0).all(), "the teacher is a stop-gradient target: detach it"
assert s.grad is not None and s.grad.abs().sum() > 0, "the student must get a gradient"

# The center moved towards the mean TEACHER logits, after the loss used the old center.
assert torch.allclose(dino.center, 0.9 * c0 + 0.1 * teacher_out.mean(0, keepdim=True), atol=1e-6)
assert not dino.center.requires_grad, "update the center under torch.no_grad()"

# Centering acts on the teacher LOGITS: shifting the logits and the center together changes nothing.
shift = torch.linspace(-2.0, 3.0, K)
a, b = DINOLoss(K, ncrops), DINOLoss(K, ncrops)
with torch.no_grad():
    a.center.copy_(c0)
    b.center.copy_(c0 + shift)
assert torch.allclose(a(student_out, teacher_out), b(student_out, teacher_out + shift), atol=1e-5)
print(f"DINO loss {loss.item():.4f}, averaged over {2 * (ncrops - 1)} (teacher, student) crop pairs")
print("DINOLoss tests passed")

In [ ]:
def ema_update_(teacher: nn.Module, student: nn.Module, momentum: float) -> None:
    """
    In-place EMA of the student's PARAMETERS into the teacher's (buffers are left alone):
        theta_t <- momentum * theta_t + (1 - momentum) * theta_s

    Args:
        teacher: module updated in place (same architecture as `student`).
        student: module whose current parameters are averaged in; not modified.
        momentum: m in [0, 1]; DINO uses 0.996, raised to 1 with a cosine schedule.
    """
    # TODO: outside autograd, update every teacher parameter in place
    raise NotImplementedError

In [ ]:
torch.manual_seed(2)
student = nn.Sequential(nn.Linear(4, 8), nn.BatchNorm1d(8), nn.ReLU(), nn.Linear(8, 3))
teacher = copy.deepcopy(student)
for p in teacher.parameters():
    p.requires_grad_(False)              # as in DINO: the teacher gets no gradients
with torch.no_grad():
    for p in student.parameters():
        p.add_(torch.randn_like(p))      # pretend the optimizer moved the student...
student(torch.randn(16, 4) * 3 + 1)      # ...and a train-mode forward moved its BN statistics

t_before = [p.detach().clone() for p in teacher.parameters()]
t_buffers = [b.clone() for b in teacher.buffers()]
param_ids = [id(p) for p in teacher.parameters()]
ema_update_(teacher, student, 0.9)
for p, p0, q in zip(teacher.parameters(), t_before, student.parameters()):
    assert torch.allclose(p, 0.9 * p0 + 0.1 * q.detach(), atol=1e-6), "teacher <- m*teacher + (1-m)*student"
    assert p.grad_fn is None and not p.requires_grad, "no autograd history on the teacher"
assert [id(p) for p in teacher.parameters()] == param_ids, "update the existing Parameters in place"
assert all(torch.equal(b, b0) for b, b0 in zip(teacher.buffers(), t_buffers)), "parameters only"
print("ema_update_ tests passed")

In [ ]:
# Look: why DINO needs BOTH centering and sharpening. A tiny DINO run on 10 Gaussian clusters
# (2 noisy views per sample, no local crops) using your DINOLoss and ema_update_. ~2 s on a CPU.
def tiny_dino(center=True, teacher_temp=0.04, steps=400, K=64):
    torch.manual_seed(0)
    protos = torch.randn(10, 16)                                  # 10 "classes"
    student = nn.Sequential(nn.Linear(16, 64), nn.GELU(), nn.Linear(64, K))
    teacher = copy.deepcopy(student).requires_grad_(False)
    # center_momentum = 1 keeps the center at 0 forever, i.e. no centering
    crit = DINOLoss(K, 2, teacher_temp=teacher_temp, center_momentum=0.9 if center else 1.0)
    opt = torch.optim.SGD(student.parameters(), lr=0.03, momentum=0.9)
    for _ in range(steps):
        x = protos[torch.randint(0, 10, (64,))]
        views = torch.cat([x + 0.7 * torch.randn_like(x), x + 0.7 * torch.randn_like(x)])
        with torch.no_grad():
            t_out = teacher(views)
        loss = crit(student(views), t_out)
        opt.zero_grad()
        loss.backward()
        opt.step()
        ema_update_(teacher, student, 0.99)
    with torch.no_grad():
        out = teacher(protos.repeat_interleave(50, 0) + 0.7 * torch.randn(500, 16))
        p = F.softmax((out - crit.center) / teacher_temp, dim=-1)
        entropy = -(p * p.clamp_min(1e-12).log()).sum(-1).mean().item()
    return entropy, out.argmax(-1).unique().numel()


print(f"K = 64 prototypes (uniform entropy log K = {math.log(64):.2f}), 10 clusters in the data")
for name, kw in [("centering + sharpening", {}),
                 ("sharpening only (no centering)", dict(center=False)),
                 ("centering only (teacher_temp = 0.1)", dict(teacher_temp=0.1))]:
    entropy, used = tiny_dino(**kw)
    print(f"  {name:36s} teacher entropy {entropy:.2f}   prototypes used (argmax) {used}")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("self-supervised-losses", nt_xent_loss=nt_xent_loss, DINOLoss=DINOLoss, ema_update_=ema_update_)